# Variational Autoencoder for the MINIST Dataset

In [ ]:
# Deep learning and neural networks
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# Loading and transforming the MINIST dataset
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import save_image

# VAE model
from vae import VAE

# General-purpose libraries
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

### Downloading the dataset

The MINST dataset is a collection of handwritten digits that is commonly used for training various image processing systems. It contains 60,000 training images and 10,000 testing images. Each image is a 28x28 grayscale image.

In this part, we will download the dataset using the `torchvision` library.

In [ ]:
trainset =  torchvision.datasets.MNIST(root='./data', train=True,   download=True, transform=transforms.ToTensor())
testset =   torchvision.datasets.MNIST(root='./data', train=False,  download=True, transform=transforms.ToTensor())

# Training set transformed into numpy arrays for easier manipulation
x_train = trainset.data.numpy()
y_train = trainset.targets.numpy()

### Training and Validation Sets

For the training set, only the first 1000 samples of each class (0-9) are selected to create a balanced validation set with 1000 samples per class. This results in a total of 10000 samples for the validation set.

For the validation set, only the first 1000 samples are selected.

In [ ]:
# Creating a balanced validation set with 1000 samples per class

idx = []
for j in range(10):
    # It selects the first 1000 indices of the class j from the training set
    # to create a balanced validation set with 1000 samples per class.
    indices_clase = np.where(y_train == j)[0][:1000]
    idx.extend(indices_clase)

x_train_pre  = x_train[idx]
y_train      = y_train[idx]

In [ ]:
# Creating the validation set with 1000 samples

x_val_pre    = testset.data[:1000].numpy()
y_val        = testset.targets[:1000].numpy()

### Image Redimensioning

Images are redimensioned from 28x28 to 14x14 to reduce the number of parameters in the model.

In [ ]:
def resize_images(images, new_size=(14, 14)):
    """
    Resize a batch of images to a new size using OpenCV.

    Parameters:
    - images: A numpy array of shape (num_images, height, width) containing the images to be resized.
    - new_size: A tuple (new_height, new_width) specifying the desired size of the output images.

    Returns:
    - A numpy array of shape (num_images, new_height, new_width) containing the resized images.
    """


    num_imgs = images.shape[0]
    new_width, new_height = new_size


    resized = np.zeros((num_imgs, new_height, new_width), dtype=np.float32)
    for i in range(num_imgs):
        resized[i] = cv2.resize(images[i].astype(np.float32), (new_width, new_height))
    return resized

NEW_WIDTH, NEW_HEIGHT = 14, 14
x_train_resized = resize_images(x_train_pre,    (NEW_WIDTH, NEW_HEIGHT))
x_val_resized = resize_images(x_val_pre,        (NEW_WIDTH, NEW_HEIGHT))

### Binarization

Binarization is applied to the images to convert them into binary images. This is done by setting a threshold value, and any pixel value above this threshold is set to 1 (white), while any pixel value below the threshold is set to 0 (black). This helps in simplifying the input data for the model.

In [ ]:
MAX_VAL = 255     # Each pixel value in the MNIST dataset is an integer between 0 and 255.
THRESHOLD = 128

x_train = np.where(x_train_resized  > THRESHOLD, 1.0, 0.0).astype(np.float32)
x_val   = np.where(x_val_resized    > THRESHOLD, 1.0, 0.0).astype(np.float32)

### Creating DataLoaders

The DataLoaders are created for both the training and validation sets. The training DataLoader is set to shuffle the data, while the validation DataLoader is not shuffled.

In [ ]:
BATCH_SIZE_TRAIN = 32
BATCH_SIZE_VAL = 100

# // TODO: Different

train_dataset   = TensorDataset(torch.from_numpy(x_train),  torch.from_numpy(y_train)   )
test_dataset     = TensorDataset(torch.from_numpy(x_val),    torch.from_numpy(y_val)     )

trainloader = DataLoader(train_dataset, batch_size=BATCH_SIZE_TRAIN,    shuffle=True    )
testloader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE_VAL,      shuffle=False   )

### Create the VAE

The model created in `vae.py` is used to create the Variational Autoencoder (VAE) model.
If possible, the model is moved to the GPU for faster training.

In [ ]:
model = VAE().to(device)

NameError: name 'vae' is not defined

### Defining loss, optimizer and LR scheduler



In [ ]:
def vae_loss_function(recon_x, x, mu, log_var):
    """
    Computes the loss for a Variational Autoencoder (VAE) model.

    Parameters:
    - recon_x: The reconstructed output from the VAE (after passing through the decoder).
    - x: The original input data (before passing through the encoder).
    - mu: The mean of the latent space distribution (output from the encoder).
    - log_var: The logarithm of the variance of the latent space distribution (output from the encoder).

    Returns:
    - total_loss: The total loss, which is the sum of the reconstruction loss (BCE) and the KL divergence (KLD).
    - BCE: The reconstruction loss, computed using binary cross-entropy.
    - KLD: The KL divergence, which measures how much the learned latent distribution deviates from the prior distribution (assumed to be a standard normal distribution).
    """
    x_flat = x.view(-1, NEW_WIDTH * NEW_HEIGHT)

    # // TODO: Revisar
    
    # Reconstruction loss (BCE) using binary cross-entropy
    BCE = F.binary_cross_entropy(recon_x, x_flat, reduction='sum')
    
    # KL divergence (KLD) using log_var
    KLD = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    
    return BCE + KLD, BCE, KLD